In [73]:
import pandas as pd
import geopandas as gpd
import altair as alt
import json
from shapely.geometry import shape
from shapely import wkt
from shapely.ops import linemerge, substring, unary_union
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [74]:
neigh = pd.read_csv("Data/Neigh/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)
neigh = neigh[["name", "geometry"]]
neigh["geometry"] = neigh.buffer(0)

neigh_layer = alt.Chart(neigh).mark_geoshape(
    stroke="black",
    fill="lightblue"
).encode(tooltip=['name'])



In [75]:
nodes = pd.read_csv("Nodes/BUS.csv")
nodes['geometry']= nodes['geometry'].apply(wkt.loads)
nodes = gpd.GeoDataFrame(nodes, geometry='geometry', crs="EPSG:4326")

In [76]:
nodes[nodes['linia'] == '197']

,id,stop_id,name,linia,stop_type,geometry


In [77]:
with open('data/Bus/AMB.json') as f:
    data = json.load(f)


traj_amb = []

for collection in data:
    for feature in collection["features"]:
        props = feature["properties"]
        geom = feature["geometry"]
        
        if geom["type"] != "Point":
            traj_amb.append({
                "linia": props["LINEA"],
                "sentit": props["SENTIDO"],
                "geometry": shape(geom)
            })

geo_df_traj_amb = gpd.GeoDataFrame(traj_amb, geometry="geometry", crs="EPSG:4326")
#geo_df_traj_amb = (geo_df_traj_amb.groupby("linia", as_index=False).agg(geometry=("geometry", lambda s: unary_union(list(s)))))

geo_df_traj_amb = gpd.GeoDataFrame(geo_df_traj_amb,geometry="geometry",crs="EPSG:4326")

buffered = neigh.copy()
buffered.to_crs('EPSG:25831', inplace=True)
buffered["geometry"] = buffered.geometry.buffer(2500)
boundary = buffered.union_all()
geo_df_traj_amb.to_crs('EPSG:25831', inplace=True)
geo_df_traj_amb["inside_ratio"] = (
    geo_df_traj_amb.geometry.intersection(boundary).length
    / geo_df_traj_amb.geometry.length
)

geo_df_traj_amb = geo_df_traj_amb[
    geo_df_traj_amb["inside_ratio"] >= 0.60   # keep lines with >=20% inside
]


geo_df_traj_amb.to_crs('EPSG:4326', inplace=True)
geo_df_traj_amb.sort_values(by='inside_ratio', inplace=True)
geo_df_traj_amb = geo_df_traj_amb[geo_df_traj_amb['linia'].isin(nodes['linia'].unique())]
geo_df_traj_amb.sort_values(by='inside_ratio', inplace=True)
print(len(geo_df_traj_amb), "lines in AMB bus network")
geo_df_traj_amb = geo_df_traj_amb[~geo_df_traj_amb["linia"].str.startswith("N")] # treure bus nits
geo_df_traj_amb = geo_df_traj_amb[geo_df_traj_amb["linia"].isin(['A1','A2','MB3','PR4','JT','135','198','197']) == False] # treure bus aerop
geo_df_traj_amb = geo_df_traj_amb[~((geo_df_traj_amb["linia"] == '88') & (geo_df_traj_amb["sentit"].isin(["Sentido Cal Truco - Edifici Area PIF", "Sentido Metro Paral?lel Mal"])))]
geo_df_traj_amb = geo_df_traj_amb[~((geo_df_traj_amb["linia"] == '127') & (geo_df_traj_amb["sentit"] == 'Sentido Favencia - Pablo Iglesias Mal'))]

geo_df_traj_amb['sentit'] = (
    geo_df_traj_amb.groupby('linia')
    .cumcount()
    .map({0: 'Anada', 1: 'Tornada'})
)
geo_df_traj_amb.drop(columns=['inside_ratio'], inplace=True)

244 lines in AMB bus network


In [78]:
geo_df_traj_amb

,linia,sentit,geometry
87,PR3,Anada,"LINESTRING (2.09041 41.33098, 2.09041 41.33095..."
98,78,Anada,"LINESTRING (2.04758 41.37601, 2.04753 41.37594..."
181,PR3,Tornada,"LINESTRING (2.0842 41.28086, 2.08421 41.28085,..."
383,78,Tornada,"LINESTRING (2.14264 41.37997, 2.14263 41.37997..."
159,B24,Anada,"LINESTRING (2.17491 41.39018, 2.1749 41.39015,..."
...,...,...,...
318,13,Tornada,"LINESTRING (2.16007 41.37635, 2.16028 41.37646..."
91,199,Anada,"LINESTRING (2.17945 41.46154, 2.18071 41.46093..."
230,H4,Tornada,"LINESTRING (2.11118 41.38567, 2.11117 41.38567..."
21,LH1,Anada,"LINESTRING (2.1113 41.34166, 2.11131 41.34168,..."


In [79]:
with open('data/Bus/bus_trajectories.json') as f:
    data = json.load(f)
df_bus_traj_tmb = []
for stop in data['features']:
    properties = stop.get('properties', {})
    id_recorregut = properties.get('ID_RECORREGUT')
    nom_linia = properties.get('NOM_LINIA')
    longitud = properties.get('LONGITUD')
    sentit = properties.get('DESC_SENTIT')
    paquet = properties.get('TIPUS_PAQUET')
    
    df_bus_traj_tmb.append({
            'linia': nom_linia,
            'sentit': sentit,
            'paquet': paquet,
            'geometry': shape(stop['geometry'])
        })
geo_df_bus_traj_tmb = gpd.GeoDataFrame(df_bus_traj_tmb,crs="EPSG:4326")
geo_df_bus_traj_tmb = geo_df_bus_traj_tmb[geo_df_bus_traj_tmb['paquet'] == '1']
geo_df_bus_traj_tmb.drop(columns=['paquet'], inplace=True)


In [80]:
lines_tmb = set(geo_df_bus_traj_tmb['linia'].unique())
lines_amb = set(geo_df_traj_amb['linia'].unique())
geo_df_traj_amb = geo_df_traj_amb[~geo_df_traj_amb['linia'].isin(lines_tmb)]
geo_df_traj_amb = pd.concat([geo_df_bus_traj_tmb, geo_df_traj_amb], ignore_index=True)


In [81]:
lines = alt.Chart(geo_df_traj_amb[(geo_df_traj_amb['linia'] == '198')]).mark_geoshape(
    filled=False, 
    strokeWidth=4
).encode(
    color=alt.Color('tram:N', legend=alt.Legend(title="Metro Segments")),
    tooltip=['tram:N']
)

points = alt.Chart(nodes[nodes['linia'] == '198']).mark_geoshape(size=0, color='red').encode(tooltip=['name:N'])
plot = (lines).project('mercator').properties(width=800, height=600)
neigh_layer + plot
    

alt.LayerChart(...)

In [82]:
geo_df_traj_amb['linia'].value_counts()

linia
D20    2
129    2
182    2
180    2
175    2
      ..
59     2
86     1
LH2    1
199    1
LH1    1
Name: count, Length: 138, dtype: int64

In [83]:
import pandas as pd
import geopandas as gpd
from shapely.ops import linemerge, substring
from shapely.geometry import LineString, MultiLineString


def create_segments_for_trajectory(trajectory_row, assigned_stops):
    """
    Segments a single trajectory using a specific list of stops.
    """
    line_geom = trajectory_row.geometry

    if isinstance(line_geom, MultiLineString):
        line_geom = linemerge(line_geom)  # Merge into a single LineString if possible


    line_id = trajectory_row['linia']
    
    # 1. Project stops and get distances
    stop_data = []
    for _, stop in assigned_stops.iterrows():
        # Project stop onto the line
        proj_dist = line_geom.project(stop.geometry)
        stop_data.append({
            'name': stop['name'],
            'id': stop['id'],
            'dist': proj_dist
        })
    
    # 2. Sort stops by distance along line
    # Remove duplicates (stops that project to the exact same meter)
    sorted_stops = sorted(stop_data, key=lambda x: x['dist'])
    
    # 3. Create segments
    segments = []
    for i in range(len(sorted_stops) - 1):
        s1 = sorted_stops[i]
        s2 = sorted_stops[i+1]
        
        # Skip zero-length segments
        if abs(s2['dist'] - s1['dist']) < 0.1: # 10cm threshold
            continue
            
        seg_geom = substring(line_geom, s1['dist'], s2['dist'])
        
        segments.append({
            'origen': s1['id'],
            'dest': s2['id'],
            'tram': f"{s1['name']} - {s2['name']}",
            'linia': line_id,
            'type': 'Bus',
            'sentit': trajectory_row['sentit'],
            'geometry': seg_geom
        })
        
    return segments

# --- MAIN WORKFLOW ---

# 1. Project to Metric CRS (EPSG:25831 is for Barcelona/Spain UTM 31N)
# Use a metric system to ensure "distance" is in meters.
geo_df_traj_amb_m = geo_df_traj_amb.to_crs("EPSG:25831" )
nodes_m = nodes.to_crs("EPSG:25831")

all_segments = []

# 2. Iterate by Line ID
for line_id in geo_df_traj_amb_m['linia'].unique():
        
    
    print(f"Processing Line: {line_id}")
    
    line_trajs = geo_df_traj_amb_m[geo_df_traj_amb_m['linia'] == line_id]
    line_stops = nodes_m[nodes_m['linia'] == line_id]

    if line_id in ['150','198','182','86','197','LH2','180','199','135','LH1']: 
        all_segments.extend(create_segments_for_trajectory(line_trajs.iloc[0], line_stops))
        continue


    # 3. Separate stops into two groups based on proximity to trajectories
    traj_0 = line_trajs.iloc[0]
    traj_1 = line_trajs.iloc[1]
    
    stops_group_0 = []
    stops_group_1 = []
    
    for _, stop in line_stops.iterrows():
        d0 = traj_0.geometry.distance(stop.geometry)
        d1 = traj_1.geometry.distance(stop.geometry)
        
        # Assign to the closer line
        if d0 < d1:
            stops_group_0.append(stop)
        else:
            stops_group_1.append(stop)
            
    # Convert lists back to DataFrames
    stops_df_0 = gpd.GeoDataFrame(stops_group_0, crs="EPSG:25831") if stops_group_0 else None
    stops_df_1 = gpd.GeoDataFrame(stops_group_1, crs="EPSG:25831") if stops_group_1 else None
    
    # 4. Generate segments for both directions
    if stops_df_0 is not None:
        all_segments.extend(create_segments_for_trajectory(traj_0, stops_df_0))
    if stops_df_1 is not None:
        all_segments.extend(create_segments_for_trajectory(traj_1, stops_df_1))

# 5. Final Result
bus_edges = gpd.GeoDataFrame(all_segments, crs="EPSG:25831" )

bus_edges = bus_edges.to_crs("EPSG:4326")

print(f"Success! Created {len(bus_edges)} segments.")

Processing Line: D20
Processing Line: D40
Processing Line: D50
Processing Line: H2
Processing Line: H4
Processing Line: H6
Processing Line: H8
Processing Line: H10
Processing Line: H12
Processing Line: H14
Processing Line: H16
Processing Line: V1
Processing Line: V3
Processing Line: V5
Processing Line: V7
Processing Line: V9
Processing Line: V11
Processing Line: V13
Processing Line: V15
Processing Line: V17
Processing Line: V19
Processing Line: V21
Processing Line: V23
Processing Line: V25
Processing Line: V27
Processing Line: V29
Processing Line: V31
Processing Line: V33
Processing Line: X1
Processing Line: X2
Processing Line: X3
Processing Line: 6
Processing Line: 7
Processing Line: 13
Processing Line: 19
Processing Line: 21
Processing Line: 22
Processing Line: 23
Processing Line: 24
Processing Line: 27
Processing Line: 33
Processing Line: 34
Processing Line: 39
Processing Line: 46
Processing Line: 47
Processing Line: 52
Processing Line: 54
Processing Line: 55
Processing Line: 59
Pro

In [84]:
bus_edges['linia'].nunique()

124

PR3, M27, 65, 34 (només lúltim punt), 114, 115 (i falten parades), 121, 127 (disjoint), 133, 180 es raro crec q falten punts, 196, 197, 34 (nms lultim), 39,52, 59, 65
7, 86, 87, b14, b15, b18, b23, b25, b81, h10, h12, h14, lh1, lh2, m27, v11, v17, v25, v3, v31, v5 (nomes ultima), v7, v9, x1

PR3, M27, 34 (nms lultim), 39,52, 59, 65
7, 86, 87, b14, b15, b18, b23, b25, b81, h10, h12, h14, lh1, lh2, m27, v11, v17, v25, v3, v31, v5 (nomes ultima), v7, v9, x1

In [85]:
line_options = sorted(bus_edges['linia'].dropna().unique())

line_select = alt.selection_point(
    fields=['linia'],
    bind=alt.binding_select(options=line_options, name='Línia: '),
    value=[{'linia': line_options[0]}]
)

lines = alt.Chart(bus_edges).transform_filter(line_select).mark_geoshape(
    filled=False,
    strokeWidth=4,
    color='steelblue'
).encode(
    tooltip=['tram:N', 'origen:N', 'dest:N']
)

points = alt.Chart(nodes).transform_filter(line_select).mark_geoshape(
    size=0,
    color='red'
).encode(
    tooltip=['name:N', 'stop_id:N']
)

plot = (lines + points).add_params(line_select).project('mercator').properties(
    width=800,
    height=600,
    title='Interactive bus line view'
)
plot

alt.LayerChart(...)

In [86]:
lines = alt.Chart(bus_edges[(bus_edges['linia'] == 'X1') & (bus_edges['sentit'] == 'Tornada')]).mark_geoshape(
    filled=False, 
    strokeWidth=4
).encode(
    color=alt.Color('tram:N', legend=alt.Legend(title="Metro Segments")),
    tooltip=['tram:N','origen:N', 'dest:N']
)

points = alt.Chart(nodes[nodes['linia'] == 'X1']).mark_geoshape(size=0, color='red').encode(tooltip=['name:N', 'stop_id:N'])
plot = (lines + points).project('mercator').properties(width=800, height=600)
plot

alt.LayerChart(...)

In [87]:
bus_edges.drop(columns=['sentit'], inplace=True)
bus_edges.to_crs('EPSG:25831', inplace=True)
bus_edges['length'] = bus_edges['geometry'].length 
bus_edges['speed'] = 12 /3.6 # 22 kmh to ms like in the paper
bus_edges['time'] = (bus_edges['length'] / bus_edges['speed']) /60
bus_edges['directed'] = True
bus_edges.to_crs('EPSG:4326', inplace=True)
bus_edges

,origen,dest,tram,linia,type,geometry,length,speed,time,directed
0,B-D20-1284,B-D20-1282,Hospital del Mar - Platja de la Barceloneta,D20,Bus,"LINESTRING (2.19464 41.38316, 2.19288 41.38029)",350.355905,3.333333,1.751780,True
1,B-D20-1282,B-D20-1604,Platja de la Barceloneta - Pg Marítim - Pepe R...,D20,Bus,"LINESTRING (2.19288 41.38029, 2.19217 41.37914)",141.475789,3.333333,0.707379,True
2,B-D20-1604,B-D20-3348,Pg Marítim - Pepe Rubianes - Pepe Rubianes,D20,Bus,"LINESTRING (2.19217 41.37914, 2.19208 41.379, ...",228.766832,3.333333,1.143834,True
3,B-D20-3348,B-D20-955,Pepe Rubianes - Pg Joan de Borbó,D20,Bus,"LINESTRING (2.1897 41.3784, 2.18826 41.37809, ...",353.731837,3.333333,1.768659,True
4,B-D20-955,B-D20-1164,Pg Joan de Borbó - Pla de Palau - Pl Pau Vila,D20,Bus,"LINESTRING (2.18739 41.37999, 2.187 41.38097, ...",378.388903,3.333333,1.891945,True
...,...,...,...,...,...,...,...,...,...,...
5528,B-LH1-100754,B-LH1-100757,Mare de Déu de Bellvitge - Trav. Industrial - ...,LH1,Bus,"LINESTRING (2.10458 41.35055, 2.10636 41.34888...",353.353034,3.333333,1.766765,True
5529,B-LH1-100757,B-LH1-109282,Poliesportiu Municipal Bellvitge - Campus Univ...,LH1,Bus,"LINESTRING (2.10678 41.34789, 2.10678 41.34789...",167.079814,3.333333,0.835399,True
5530,B-LH1-109282,B-LH1-107491,Campus Universitari Bellvitge - Camí Pau Redó ...,LH1,Bus,"LINESTRING (2.10737 41.34648, 2.10742 41.34647...",527.435548,3.333333,2.637178,True
5531,B-LH1-107491,B-LH1-112416,Camí Pau Redó - Institut Català d'Oncologia - ...,LH1,Bus,"LINESTRING (2.11082 41.34319, 2.11083 41.34316...",190.127828,3.333333,0.950639,True


In [88]:
bus_edges.to_csv("Edges/Bus.csv")